# Balanced cluster sample: 100k users each from Western, Southern, Eastern

Builds the human response set used as ground truth for the LLM preference study.

**Output:** one CSV with the responses of **300,000 unique users**: 100,000 randomly sampled from each of the paper's three moral clusters. Each user contributes **10 complete dilemmas**, 2 each of Utilitarian, Gender, Fitness, Age and Species, with both outcome rows per dilemma (**20 rows per user**). Social Status and Random dilemmas are excluded.

| | |
|---|---|
| Source responses | `SharedResponsesFullFirstSessions.csv` (first completed session per user) |
| Cluster labels | `country_cluster_map.csv` (Awad et al. 2018, Fig 3 clusters) |
| Output folder | `data_raw_output/` |

Related docs: [`docs/scenarios_and_dilemmas.md`](../docs/scenarios_and_dilemmas.md) · [`docs/csv/SharedResponses.md`](../docs/csv/SharedResponses.md) · [`docs/csv/country_cluster_map.md`](../docs/csv/country_cluster_map.md)

## 1. Method

1. **Load clusters.** 130 countries → `0 = Western`, `1 = Eastern`, `2 = Southern`. Users from countries outside these 130 have no cluster and are excluded.
2. **Pass 1 over the responses (streamed).** For every `UserID`, record their country and how many rows they have.
3. **Scope.** Keep users who appear under a single country that is in the cluster map.
4. **Completeness grid (streamed).** For every user, record which of their 13 dilemma positions still have both outcome rows, and the scenario type of each.
5. **Eligibility.** A user is eligible if all **10** dilemmas of the kept types (Utilitarian, Gender, Fitness, Age, Species) are complete.
6. **Sample.** Within each cluster, draw **100,000 unique users** at random with a fixed seed (simple random sampling, so large countries such as the US keep their natural share).
7. **Pass 2 over the responses (streamed).** Write only the kept-type rows of each sampled user to one CSV, adding the cluster label.
8. **Validate.** Users per cluster, 20 rows and 10 dilemmas per user, one spared outcome per dilemma, 2 dilemmas of each kept type, and the country mix inside each cluster.

### Why Social Status and Random are excluded

- **Random** dilemmas don't test a single dimension, so they can't serve as labelled items.
- **Social Status:** the authors removed most Social Status rows from the shared file. About 97% of users have at least one Social Status dilemma that is half-missing or gone. Requiring it would leave only ~3k Eastern users. The other 5 types are almost always intact.

### Expectations

| Cluster | Users in scope (approx.) | Sampled |
|---|---|---|
| Western | ~1.2M | 100,000 |
| Southern | ~304k | 100,000 |
| Eastern | ~103k | 100,000 (almost all of them) |

Expected output size: 300,000 users × 20 rows = **6M rows**.

**Runtime:** each pass reads the ~4.8 GB source file, so expect roughly 5–10 minutes per pass (three passes in total).

## 2. Setup

`UserID` and `ExtendedSessionID` are read as **strings**. As floats, the long numeric IDs lose precision and different users can collapse into one ID.

In [10]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MM = ROOT / "3hvt2-osfstorage-archive" / "Datasets" / "Moral Machine Data"
SOURCE = MM / "SharedResponsesFullFirstSessions.csv" / "SharedResponsesFullFirstSessions.csv"
CLUSTER_MAP = MM / "country_cluster_map.csv" / "country_cluster_map.csv"

OUT_DIR = ROOT / "data_raw_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_RESPONSES = OUT_DIR / "balanced_cluster_responses_300k.csv"
OUT_USERS = OUT_DIR / "balanced_cluster_users_300k.csv"

N_PER_CLUSTER = 100_000
KEEP_TYPES = ["Utilitarian", "Gender", "Fitness", "Age", "Species"]  # Social Status and Random excluded
DILEMMAS_PER_USER = 2 * len(KEEP_TYPES)  # 10
ROWS_PER_USER = 2 * DILEMMAS_PER_USER    # 20
SEED = 42
CHUNKSIZE = 1_000_000

CLUSTER_NAMES = {0: "Western", 1: "Eastern", 2: "Southern"}

ID_DTYPES = {"UserID": "string", "ExtendedSessionID": "string", "ResponseID": "string", "UserCountry3": "string"}

for p in (SOURCE, CLUSTER_MAP):
    assert p.exists(), f"Missing: {p}"
print(f"Source: {SOURCE.stat().st_size / 1e9:.2f} GB")
print(f"Output: {OUT_RESPONSES}")

Source: 4.82 GB
Output: e:\personal\Mres\mechanistic interpretability\literature review\moral data\data_raw_output\balanced_cluster_responses_300k.csv


## 3. Load the cluster map

In [11]:
clusters = pd.read_csv(CLUSTER_MAP, dtype={"ISO3": "string"})
clusters["ClusterName"] = clusters["Cluster"].map(CLUSTER_NAMES)
iso_to_cluster = dict(zip(clusters["ISO3"], clusters["Cluster"]))

clusters["ClusterName"].value_counts().rename("countries").to_frame()

,countries
ClusterName,
Western,67
Southern,35
Eastern,28


## 4. Pass 1: profile every user

Only `UserID` and `UserCountry3` are read, which keeps this pass light. For each user we count rows and collect the countries they appear under. A blank country is stored as an empty string; those users match no cluster and are excluded in the next step.

In [12]:
row_counts = Counter()
user_country = {}
multi_country = set()
n_rows = 0

reader = pd.read_csv(
    SOURCE,
    usecols=["UserID", "UserCountry3"],
    dtype=ID_DTYPES,
    chunksize=CHUNKSIZE,
)

for i, chunk in enumerate(reader, start=1):
    n_rows += len(chunk)
    chunk = chunk.dropna(subset=["UserID"])
    chunk["UserCountry3"] = chunk["UserCountry3"].fillna("")
    row_counts.update(chunk["UserID"].tolist())

    pairs = chunk.drop_duplicates()
    for uid, iso in zip(pairs["UserID"], pairs["UserCountry3"]):
        prev = user_country.setdefault(uid, iso)
        if prev != iso:
            multi_country.add(uid)

    if i % 5 == 0:
        print(f"... {n_rows:,} rows, {len(row_counts):,} users")

print(f"Done: {n_rows:,} rows, {len(row_counts):,} unique users, {len(multi_country):,} with >1 country")

... 5,000,000 rows, 1,575,438 users
... 10,000,000 rows, 1,621,728 users
... 15,000,000 rows, 1,622,019 users
... 20,000,000 rows, 1,622,019 users
... 25,000,000 rows, 1,622,019 users
... 30,000,000 rows, 1,622,019 users
... 35,000,000 rows, 1,622,019 users
Done: 38,078,300 rows, 1,622,019 unique users, 5,108 with >1 country


## 5. Users in scope per cluster

A user is in scope if they appear under one country and that country is in the cluster map. Completeness is checked in 5b, and the final eligible list is built in 5c.

In [14]:
users = pd.DataFrame({
    "UserID": list(row_counts.keys()),
    "n_rows": list(row_counts.values()),
})
users["UserCountry3"] = users["UserID"].map(user_country)
users["Cluster"] = users["UserCountry3"].map(iso_to_cluster)
users["ClusterName"] = users["Cluster"].map(CLUSTER_NAMES)

exclusions = pd.Series({
    "all users": len(users),
    "more than one country": int(users["UserID"].isin(multi_country).sum()),
    "country not in cluster map": int(users["Cluster"].isna().sum()),
}, name="users")

scope_users = users[~users["UserID"].isin(multi_country) & users["Cluster"].notna()]

display(exclusions.to_frame())
scope_users["ClusterName"].value_counts().rename("users in scope").to_frame()

,users
all users,1622019
more than one country,5108
country not in cluster map,5511


,users in scope
ClusterName,
Western,1205451
Southern,304325
Eastern,102903


## 5b. Which dilemmas survived for each user

Most users don't have all 26 rows. The original authors removed some individual outcome rows during cleaning, so many dilemmas now have only one of their two rows, and some are gone entirely.

This pass builds a grid of **users × 13 dilemma positions** (`ScenarioOrder`). Each cell holds how many rows of that dilemma are still in the file:

| Rows in a slot | Meaning |
|---|---|
| 2 | **complete** dilemma (both outcomes present, usable) |
| 1 | **half-missing**: one outcome row was removed, so the choice can't be read |
| 0 | **missing**: the whole dilemma is absent |

It also records each slot's scenario type, so only the kept types (Utilitarian, Gender, Fitness, Age, Species) count toward completeness. Social Status and Random slots are ignored. The grid is stored as compact numpy arrays (about 50 MB), not per-row Python objects.

**Runtime:** another full read of the source file, about 5–10 minutes.

In [15]:
TYPE_CODES = {"Utilitarian": 1, "Gender": 2, "Fitness": 3, "Age": 4, "Social Status": 5, "Species": 6, "Random": 7}
TYPE_NAMES = {0: "(unknown)", **{v: k for k, v in TYPE_CODES.items()}}
N_SLOTS = 13

uid_list = list(row_counts.keys())
uid_index = {u: i for i, u in enumerate(uid_list)}
n_users = len(uid_list)

# columns 1..13 = ScenarioOrder; column 0 is unused, column 14 collects out-of-range orders
slot_rows = np.zeros((n_users, N_SLOTS + 2), dtype=np.uint8)
slot_type = np.zeros((n_users, N_SLOTS + 2), dtype=np.int8)

reader = pd.read_csv(
    SOURCE,
    usecols=["UserID", "ScenarioOrder", "ScenarioTypeStrict", "ScenarioType"],
    dtype={"UserID": "string", "ScenarioTypeStrict": "string", "ScenarioType": "string"},
    chunksize=CHUNKSIZE,
)

for i, chunk in enumerate(reader, start=1):
    chunk = chunk.dropna(subset=["UserID", "ScenarioOrder"])
    idx = chunk["UserID"].map(uid_index).to_numpy(dtype=np.int64)
    order = chunk["ScenarioOrder"].to_numpy(dtype=np.int64)
    order = np.where((order >= 1) & (order <= N_SLOTS), order, N_SLOTS + 1)

    stype = (
        chunk["ScenarioTypeStrict"].fillna(chunk["ScenarioType"])
        .map(TYPE_CODES).fillna(0).to_numpy(dtype=np.int8)
    )

    np.add.at(slot_rows, (idx, order), 1)
    slot_type[idx, order] = stype

    if i % 5 == 0:
        print(f"... chunk {i}")

rows_grid = slot_rows[:, 1 : N_SLOTS + 1]
type_grid = slot_type[:, 1 : N_SLOTS + 1]
is_random = type_grid == TYPE_CODES["Random"]

is_kept = np.isin(type_grid, [TYPE_CODES[t] for t in KEEP_TYPES])

complete_nonrandom = ((rows_grid == 2) & ~is_random).sum(axis=1)
complete_kept = ((rows_grid == 2) & is_kept).sum(axis=1)
half_missing = (rows_grid == 1).sum(axis=1)
fully_missing = (rows_grid == 0).sum(axis=1)

print(f"Done. Users: {n_users:,} | rows with ScenarioOrder outside 1-13: {int(slot_rows[:, N_SLOTS + 1].sum()):,}")

... chunk 5
... chunk 10
... chunk 15
... chunk 20
... chunk 25
... chunk 30
... chunk 35
Done. Users: 1,622,019 | rows with ScenarioOrder outside 1-13: 24


### Users below a threshold, and what they are missing

Set `THRESHOLD` to the minimum number of **complete dilemmas of the 5 kept types** a user needs (the maximum is 10). This cell is a diagnostic only; eligibility in 5c always requires all 10. The cell shows:

1. How many users have each number of complete kept-type dilemmas.
2. Per cluster, how many users pass vs. fall below the threshold. Only users with one clustered country are counted.
3. A table of below-threshold users listing which dilemma positions are **half-missing** (one row left) and which are **missing** (no rows), with the scenario type of each half-missing slot. A fully missing slot has no rows, so its type can't be known.

The full below-threshold table is also saved to `data_raw_output/users_below_threshold.csv`.

To inspect a single user's 13 slots, call `show_user("<UserID>")`.

In [16]:
THRESHOLD = DILEMMAS_PER_USER  # minimum complete kept-type dilemmas per user (max 10)
SHOW_N = 20     # below-threshold users to display

grid_users = pd.DataFrame({
    "UserID": uid_list,
    "complete_kept": complete_kept,
    "half_missing": half_missing,
    "fully_missing": fully_missing,
})
grid_users["UserCountry3"] = grid_users["UserID"].map(user_country)
grid_users["ClusterName"] = grid_users["UserCountry3"].map(iso_to_cluster).map(CLUSTER_NAMES)
in_scope = grid_users["ClusterName"].notna() & ~grid_users["UserID"].isin(multi_country)

# 1) distribution
print("Users by number of complete kept-type dilemmas:")
display(grid_users["complete_kept"].value_counts().sort_index().rename("users").to_frame())

# 2) pass / below per cluster
scoped = grid_users[in_scope]
summary = (
    scoped.assign(passes=scoped["complete_kept"] >= THRESHOLD)
    .groupby("ClusterName")["passes"]
    .agg(passing="sum", total="size")
)
summary["below"] = summary["total"] - summary["passing"]
print(f"\nThreshold = {THRESHOLD} complete kept-type dilemmas")
display(summary[["passing", "below", "total"]])

# 2b) users passing at every threshold, per cluster
counts = pd.crosstab(scoped["complete_kept"], scoped["ClusterName"])
passing_by_threshold = counts.iloc[::-1].cumsum().iloc[::-1]
passing_by_threshold.index.name = "threshold (>=)"
print(f"\nUsers passing at each threshold (need {N_PER_CLUSTER:,} per cluster):")
display(passing_by_threshold)

# 3) which slots are missing for below-threshold users
below_mask = in_scope.to_numpy() & (complete_kept < THRESHOLD)
below_idx = np.flatnonzero(below_mask)


def slot_list(row_idx, value):
    return [k + 1 for k in np.flatnonzero(rows_grid[row_idx] == value)]


below = grid_users.iloc[below_idx].copy()
below["half_missing_slots"] = [slot_list(r, 1) for r in below_idx]
below["missing_slots"] = [slot_list(r, 0) for r in below_idx]
below["half_missing_types"] = [
    [TYPE_NAMES[int(type_grid[r, s - 1])] for s in slots]
    for r, slots in zip(below_idx, below["half_missing_slots"])
]

below_path = OUT_DIR / "users_below_threshold.csv"
below.to_csv(below_path, index=False)
print(f"\n{len(below):,} users below threshold -> {below_path}")

below.head(SHOW_N)[[
    "UserID", "UserCountry3", "ClusterName", "complete_kept",
    "half_missing_slots", "half_missing_types", "missing_slots",
]]

Users by number of complete kept-type dilemmas:


,users
complete_kept,
0,1
1,7
2,4
3,6
4,4
5,4
6,11
7,8
8,52



Threshold = 10 complete kept-type dilemmas


,passing,below,total
ClusterName,,,
Eastern,102839,64,102903
Southern,304189,136,304325
Western,1204921,530,1205451



Users passing at each threshold (need 100,000 per cluster):


ClusterName,Eastern,Southern,Western
threshold (>=),,,
0,102903,304325,1205451
1,102903,304324,1205451
2,102902,304323,1205447
3,102901,304323,1205446
4,102900,304322,1205444
5,102900,304322,1205441
6,102900,304322,1205440
7,102899,304320,1205433
8,102898,304320,1205428



730 users below threshold -> e:\personal\Mres\mechanistic interpretability\literature review\moral data\data_raw_output\users_below_threshold.csv


,UserID,UserCountry3,ClusterName,complete_kept,half_missing_slots,half_missing_types,missing_slots
376,3951873278072100,NZL,Western,9,"[8, 10]","[Species, Social Status]",[2]
3925,7296811332706420,KOR,Eastern,9,"[10, 13]","[Social Status, Species]",[9]
7063,9629920920063800,GBR,Western,9,"[1, 5, 9]","[Species, Social Status, Social Status]",[]
7947,2814233661436460,DEU,Western,9,"[2, 12]","[Social Status, Species]",[]
10932,1270017265161290,RUS,Western,9,[9],[Social Status],"[1, 3]"
17386,4047903467203950,CAN,Western,9,"[3, 10]","[Social Status, Species]",[8]
19346,3700882369660710,BRA,Western,9,"[1, 3]","[Social Status, Species]",[]
21152,9954632575031740,BRA,Western,9,[2],[Social Status],"[7, 8]"
21995,9734788495728110,GBR,Western,9,"[2, 8]","[Species, Social Status]",[]
28272,143323937602803,GBR,Western,8,[5],[Social Status],"[8, 13]"


In [17]:
STATUS = {2: "complete", 1: "half-missing (1 row)", 0: "missing"}


def show_user(uid: str) -> pd.DataFrame:
    """13-slot view for one user: rows present, status and scenario type per dilemma position."""
    r = uid_index[uid]
    return pd.DataFrame({
        "ScenarioOrder": range(1, N_SLOTS + 1),
        "rows_present": rows_grid[r],
        "status": [STATUS.get(int(v), f"{int(v)} rows") for v in rows_grid[r]],
        "scenario_type": [
            TYPE_NAMES[int(t)] if n > 0 else "(unknown)"
            for t, n in zip(type_grid[r], rows_grid[r])
        ],
    }).set_index("ScenarioOrder")


example_uid = below["UserID"].iloc[0] if len(below) else uid_list[0]
print(f"Example user: {example_uid}")
show_user(example_uid)

Example user: 3951873278072100


,rows_present,status,scenario_type
ScenarioOrder,,,
1,2,complete,Utilitarian
2,0,missing,(unknown)
3,2,complete,Gender
4,2,complete,Fitness
5,2,complete,Age
6,2,complete,Fitness
7,2,complete,Age
8,1,half-missing (1 row),Species
9,2,complete,Utilitarian


## 5c. Eligible users

A user is eligible if they are in scope (one clustered country) and all **10** dilemmas of the kept types are complete (`complete_kept == DILEMMAS_PER_USER`). Social Status and Random dilemmas are ignored, whatever their state.

In [18]:
eligible = grid_users.loc[
    in_scope & (grid_users["complete_kept"] == DILEMMAS_PER_USER),
    ["UserID", "UserCountry3", "ClusterName"],
].copy()
eligible["Cluster"] = eligible["UserCountry3"].map(iso_to_cluster).astype(int)

eligible_counts = pd.DataFrame({
    "in scope": grid_users.loc[in_scope, "ClusterName"].value_counts(),
    "eligible": eligible["ClusterName"].value_counts(),
})
eligible_counts["eligible_%"] = (100 * eligible_counts["eligible"] / eligible_counts["in scope"]).round(1)
eligible_counts["enough for sample"] = eligible_counts["eligible"] >= N_PER_CLUSTER
eligible_counts

,in scope,eligible,eligible_%,enough for sample
ClusterName,,,,
Western,1205451,1204921,100.0,True
Southern,304325,304189,100.0,True
Eastern,102903,102839,99.9,True


## 6. Sample 100,000 users per cluster

Simple random sampling without replacement, seed fixed for reproducibility. If a cluster has fewer than 100,000 eligible users, all of them are taken and a warning is printed.

In [19]:
rng = np.random.default_rng(SEED)
parts = []
for cid, name in CLUSTER_NAMES.items():
    pool = eligible[eligible["Cluster"] == cid]
    if len(pool) < N_PER_CLUSTER:
        print(f"WARNING: {name} has only {len(pool):,} eligible users; taking all of them")
        parts.append(pool)
    else:
        idx = rng.choice(len(pool), size=N_PER_CLUSTER, replace=False)
        parts.append(pool.iloc[np.sort(idx)])

sample_users = pd.concat(parts, ignore_index=True)[["UserID", "UserCountry3", "Cluster", "ClusterName"]]
assert sample_users["UserID"].is_unique

sample_users.to_csv(OUT_USERS, index=False)
print(f"Saved {len(sample_users):,} users -> {OUT_USERS}")
sample_users["ClusterName"].value_counts().rename("sampled users").to_frame()

Saved 300,000 users -> e:\personal\Mres\mechanistic interpretability\literature review\moral data\data_raw_output\balanced_cluster_users_300k.csv


,sampled users
ClusterName,
Western,100000
Eastern,100000
Southern,100000


### Country mix inside each sampled cluster

Check how much a single country dominates each cluster. Under simple random sampling, expect roughly 28% US in Western, 33% France in Southern and 43% Japan in Eastern.

In [20]:
mix = (
    sample_users.groupby(["ClusterName", "UserCountry3"]).size().rename("users").reset_index()
    .merge(clusters[["ISO3", "Country"]], left_on="UserCountry3", right_on="ISO3", how="left")
)
mix["share_%"] = (100 * mix["users"] / mix.groupby("ClusterName")["users"].transform("sum")).round(1)
mix = mix.sort_values(["ClusterName", "users"], ascending=[True, False])

print("Countries per cluster:", mix.groupby("ClusterName").size().to_dict())
mix.groupby("ClusterName").head(5)[["ClusterName", "Country", "users", "share_%"]]

Countries per cluster: {'Eastern': 28, 'Southern': 35, 'Western': 67}


,ClusterName,Country,users,share_%
12,Eastern,Japan,42573,42.6
27,Eastern,Taiwan,7324,7.3
9,Eastern,India,7259,7.3
14,Eastern,South Korea,6859,6.9
7,Eastern,Hong Kong,6007,6.0
39,Southern,France,33229,33.2
60,Southern,Turkey,16886,16.9
35,Southern,Czechia,13501,13.5
43,Southern,Hungary,11570,11.6
46,Southern,Mexico,4991,5.0


## 7. Pass 2: extract the kept dilemmas of the sampled users

Only rows of the kept types are written; Social Status and Random rows are dropped. The type is `ScenarioTypeStrict`, falling back to `ScenarioType` when blank (the same rule as 5b). All 21 source columns are kept, plus `Cluster` and `ClusterName`. Rows are appended chunk by chunk, so the full 6M rows never sit in memory at once. Any existing output file is overwritten.

In [21]:
sampled_ids = set(sample_users["UserID"])
uid_to_cluster = dict(zip(sample_users["UserID"], sample_users["Cluster"]))

if OUT_RESPONSES.exists():
    OUT_RESPONSES.unlink()

n_written = 0
first = True
reader = pd.read_csv(SOURCE, dtype=ID_DTYPES, chunksize=CHUNKSIZE)

for i, chunk in enumerate(reader, start=1):
    keep = chunk[chunk["UserID"].isin(sampled_ids)]
    stype = keep["ScenarioTypeStrict"].fillna(keep["ScenarioType"])
    keep = keep[stype.isin(KEEP_TYPES)].copy()
    if len(keep):
        keep["Cluster"] = keep["UserID"].map(uid_to_cluster).astype(int)
        keep["ClusterName"] = keep["Cluster"].map(CLUSTER_NAMES)
        keep.to_csv(OUT_RESPONSES, mode="w" if first else "a", header=first, index=False)
        first = False
        n_written += len(keep)
    if i % 5 == 0:
        print(f"... chunk {i}: {n_written:,} rows written")

print(f"Done: {n_written:,} rows -> {OUT_RESPONSES}")
print(f"Expected: {len(sample_users) * ROWS_PER_USER:,}")
print(f"File size: {OUT_RESPONSES.stat().st_size / 1e9:.2f} GB")

... chunk 5: 786,675 rows written
... chunk 10: 1,572,562 rows written
... chunk 15: 2,358,915 rows written
... chunk 20: 3,145,919 rows written
... chunk 25: 3,935,635 rows written
... chunk 30: 4,724,364 rows written
... chunk 35: 5,513,828 rows written
Done: 6,000,000 rows -> e:\personal\Mres\mechanistic interpretability\literature review\moral data\data_raw_output\balanced_cluster_responses_300k.csv
Expected: 6,000,000
File size: 0.88 GB


## 8. Validate the output

Reloads only the columns needed for checks:

- every sampled user appears, with exactly 20 rows and 10 dilemmas (`ResponseID`s)
- each dilemma has exactly 2 rows and exactly one spared outcome (`Saved = 1`)
- no Social Status or Random rows remain
- 100,000 users per cluster

In [22]:
check = pd.read_csv(
    OUT_RESPONSES,
    usecols=["UserID", "ResponseID", "Saved", "ScenarioTypeStrict", "ScenarioType", "ClusterName"],
    dtype={"UserID": "string", "ResponseID": "string", "ScenarioTypeStrict": "string",
           "ScenarioType": "string", "ClusterName": "string"},
)
check["Type"] = check["ScenarioTypeStrict"].fillna(check["ScenarioType"])

per_user = check.groupby("UserID").agg(rows=("ResponseID", "size"), dilemmas=("ResponseID", "nunique"))
per_dilemma = check.groupby("ResponseID").agg(rows=("Saved", "size"), saved=("Saved", "sum"))

results = pd.Series({
    "users in output": len(per_user),
    "users expected": len(sample_users),
    f"users with {ROWS_PER_USER} rows": int((per_user["rows"] == ROWS_PER_USER).sum()),
    f"users with {DILEMMAS_PER_USER} dilemmas": int((per_user["dilemmas"] == DILEMMAS_PER_USER).sum()),
    "dilemmas total": len(per_dilemma),
    "dilemmas with 2 rows and one Saved=1": int(((per_dilemma["rows"] == 2) & (per_dilemma["saved"] == 1)).sum()),
    "rows of excluded types": int((~check["Type"].isin(KEEP_TYPES)).sum()),
}, name="value")
display(results.to_frame())

check.drop_duplicates("UserID")["ClusterName"].value_counts().rename("users").to_frame()

,value
users in output,300000
users expected,300000
users with 20 rows,300000
users with 10 dilemmas,300000
dilemmas total,3000000
dilemmas with 2 rows and one Saved=1,3000000
rows of excluded types,0


,users
ClusterName,
Southern,100000
Western,100000
Eastern,100000


### Dilemma types per user

Counted on dilemmas (one row per `ResponseID`). Every user should have exactly **2.0** of each of the 5 kept types.

In [23]:
dilemmas = check.drop_duplicates("ResponseID")
types_per_user = (
    dilemmas.groupby(["ClusterName", "Type"]).size()
    .unstack(fill_value=0)
    .div(dilemmas.groupby("ClusterName")["UserID"].nunique(), axis=0)
    .round(2)
)
types_per_user

Type,Age,Fitness,Gender,Species,Utilitarian
ClusterName,,,,,
Eastern,2.0,2.0,2.0,2.0,2.0
Southern,2.0,2.0,2.0,2.0,2.0
Western,2.0,2.0,2.0,2.0,2.0


## 9. Output files

| File | Contents |
|---|---|
| `data_raw_output/balanced_cluster_responses_300k.csv` | 20 rows per sampled user (10 complete dilemmas of the kept types): 21 source columns + `Cluster`, `ClusterName` |
| `data_raw_output/balanced_cluster_users_300k.csv` | One row per sampled user: `UserID`, `UserCountry3`, `Cluster`, `ClusterName` |
| `data_raw_output/users_below_threshold.csv` | Diagnostic from 5b: users below `THRESHOLD` and which dilemma positions they are missing |

### Column reminder

- **2 rows per dilemma**, sharing a `ResponseID`. `Saved = 1` marks the outcome the user chose to spare.
- **`ScenarioTypeStrict`** gives the character dimension: here only Utilitarian, Gender, Fitness, Age and Species. `AttributeLevel` gives this outcome's level on it.
- **Structural fields:** `Intervention` (stay/swerve), `Barrier` and `PedPed` (passengers/pedestrians), `CrossingSignal` (legality).
- **`ScenarioOrder`** is presentation position 1–13 only. It is random, so don't group preferences by it. Positions of the dropped Social Status and Random dilemmas are simply absent.

### Next step

Compute the human ground truth from this file: spare rate, majority side and AMCE for the 5 character dimensions and the 3 structural dimensions (Intervention, Relation to vehicle, Law), overall and per cluster. Social Status is not covered by this sample. See [`docs/scenarios_and_dilemmas.md`](../docs/scenarios_and_dilemmas.md) for how each dimension is filtered.